# GibbsQ Colab / Hosted Runtime Runner

This notebook is a clean runner for cloud environments such as Google Colab, Kaggle, or other hosted Jupyter platforms.

It is designed to call the existing project scripts directly, especially `scripts/execution/reproduction_pipeline.py`, instead of duplicating pipeline logic inside the notebook.

Workflow:
- locate the repository root
- install the project in the active notebook environment
- verify the main execution script exists
- dry-run the reproduction pipeline
- launch either the full pipeline or a chosen phase
- inspect generated outputs


In [ ]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print(f'Python executable: {sys.executable}')
print(f'Current working directory: {Path.cwd()}')
print(f'Running inside Colab: {IN_COLAB}')


## Optional: mount Google Drive

Run the next cell only if the repository lives on your Google Drive. If you already opened this notebook inside the repository workspace, you can skip it.


In [ ]:
if IN_COLAB:
    # Uncomment the next two lines if the repo is stored in Drive.
    # drive.mount('/content/drive')
    # print('Drive mounted at /content/drive')
    pass


In [ ]:
DEFAULT_REPO_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/GibbsQ'),
    Path('/content/drive/MyDrive/GibbsQ'),
    Path('/kaggle/working/GibbsQ'),
]

def find_repo_root(candidates: list[Path]) -> Path:
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'scripts' / 'execution' / 'reproduction_pipeline.py').is_file():
            return candidate
    raise FileNotFoundError(
        'Could not locate the GibbsQ repository root. Update DEFAULT_REPO_CANDIDATES or set REPO_ROOT manually.'
    )

REPO_ROOT = find_repo_root(DEFAULT_REPO_CANDIDATES)
PIPELINE_SCRIPT = REPO_ROOT / 'scripts' / 'execution' / 'reproduction_pipeline.py'
EXPERIMENT_RUNNER = REPO_ROOT / 'scripts' / 'execution' / 'experiment_runner.py'
OUTPUTS_DIR = REPO_ROOT / 'outputs'

os.chdir(REPO_ROOT)
print(f'Repository root: {REPO_ROOT}')
print(f'Pipeline script: {PIPELINE_SCRIPT}')
print(f'Experiment runner: {EXPERIMENT_RUNNER}')


## Install the project into the active notebook kernel

This keeps the notebook simple and portable. We install from `pyproject.toml` and then use the existing scripts exactly as the repository intends.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True, cwd=REPO_ROOT)
print('Editable install finished successfully.')


In [ ]:
assert PIPELINE_SCRIPT.is_file(), f'Missing pipeline script: {PIPELINE_SCRIPT}'
assert EXPERIMENT_RUNNER.is_file(), f'Missing experiment runner: {EXPERIMENT_RUNNER}'
print('Execution scripts are present and ready.')


## Configure the run

Change the values in the next cell to match the runtime budget you want.


In [ ]:
PIPELINE_PROGRESS = 'on'
PIPELINE_START_FROM = None  # examples: None, 'drift', 'policy'
PIPELINE_DRY_RUN = True
EXTRA_HYDRA_ARGS = [
    '--config-name', 'small',
]

print({
    'progress': PIPELINE_PROGRESS,
    'start_from': PIPELINE_START_FROM,
    'dry_run': PIPELINE_DRY_RUN,
    'extra_hydra_args': EXTRA_HYDRA_ARGS,
})


In [ ]:
def build_pipeline_command(*, dry_run: bool, start_from: str | None, progress: str, hydra_args: list[str]) -> list[str]:
    cmd = [sys.executable, str(PIPELINE_SCRIPT), '--progress', progress]
    if dry_run:
        cmd.append('--dry-run')
    if start_from:
        cmd.extend(['--start-from', start_from])
    cmd.extend(hydra_args)
    return cmd

pipeline_cmd = build_pipeline_command(
    dry_run=PIPELINE_DRY_RUN,
    start_from=PIPELINE_START_FROM,
    progress=PIPELINE_PROGRESS,
    hydra_args=EXTRA_HYDRA_ARGS,
)
print('Resolved command:')
print(' '.join(pipeline_cmd))


## Run a dry-run first

Keep `PIPELINE_DRY_RUN = True` until the resolved command looks correct for the platform.


In [ ]:
subprocess.run(pipeline_cmd, check=True, cwd=REPO_ROOT)


## Optional: run a single experiment directly

This is useful on hosted platforms when you want to validate one phase before launching the full pipeline.


In [ ]:
SINGLE_EXPERIMENT = 'policy'
SINGLE_EXPERIMENT_ARGS = ['--config-name', 'small']

single_cmd = [
    sys.executable,
    str(EXPERIMENT_RUNNER),
    '--progress', 'on',
    SINGLE_EXPERIMENT,
    *SINGLE_EXPERIMENT_ARGS,
]

print(' '.join(single_cmd))
# subprocess.run(single_cmd, check=True, cwd=REPO_ROOT)


## Inspect outputs

After a real run, this cell gives a quick view of the generated artifacts.


In [ ]:
if OUTPUTS_DIR.exists():
    files = sorted(p for p in OUTPUTS_DIR.rglob('*') if p.is_file())
    print(f'Found {len(files)} output files.')
    for path in files[:50]:
        print(path.relative_to(REPO_ROOT))
else:
    print(f'Outputs directory does not exist yet: {OUTPUTS_DIR}')
